James Caldwell <br>
October 2025 <br>

This script automates prize emails for IGOW/RaceGOW

In [53]:
import numpy as np
import pandas as pd

## Download Prize Data from Google Sheets
    # Google sheets full url: https://docs.google.com/spreadsheets/d/1p7wASAe13hgVXKNCimAjb5oJkW2To8OnCojyqOP3WJg/edit?gid=0#gid=0
sheet_id = "1p7wASAe13hgVXKNCimAjb5oJkW2To8OnCojyqOP3WJg"
gid = "0"  # the tab’s unique gid
csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"

prize_df = pd.read_csv(csv_url, skiprows=1)
open_prizes = prize_df.iloc[0:87,:] # Only grabs open prizes for now. Need to adjust based on this year's formatting


## Download Emails from Google Sheets
sheet_id = "1jvZho0IOuNjtqPmEF7Gtvi8hAcUeSaiebLWSsAArszg"
gid = "1962900089"  # the tab’s unique gid
csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
registration_df = pd.read_csv(csv_url)
registration_df['Email'] = 'charlottesville.drone@gmail.com'
callsign_and_email = registration_df[['What is your Pilot Callsign (Handle)?','Email']]


In [54]:
# Generate list of weeks for GUI dropdown
week_dropdown_list = sorted(open_prizes.iloc[:, 0].unique())



In [55]:
def get_winner_info(week):
    week_df = open_prizes[open_prizes.iloc[:, 0] == week].copy()
    for winners in week_df.iloc[:, 1]:
        print(winners)
        week_df.loc[week_df.iloc[:, 1] == winners, 'Email'] = ', '.join(
            callsign_and_email[callsign_and_email.iloc[:, 0] == winners]['Email'].tolist())
    return week_df

week = 'Track1'
winner_info = get_winner_info(week)
winner_info

Lone FPV
cybaix
Hurricane
GunjaFPV
Yamasaur
FpvGaps
JANxD
GOONZ
Rodgsilva


,Track#,Winner,Prize Sponsor,Retail Value,Prize Type,Prize Details,Unnamed: 6,Email
0,Track1,Lone FPV,ZachC82,$200.00,Giveaway,Gift Card to TinyWhoop.com,NaN,charlottesville.drone@gmail.com
1,Track1,cybaix,Tiny Whoop,$100.00,Giveaway,CASH via PayPal,NaN,charlottesville.drone@gmail.com
2,Track1,Hurricane,VIFLY + CustomFPV + The Prop Popper + Gemfan +...,$99.00,Giveaway,Whoopstor V3 + RaceGOW3 Exclusive 65mm 2-bange...,NaN,charlottesville.drone@gmail.com
3,Track1,GunjaFPV,The Prop Popper + HQProp + Gemfan,$37.00,Giveaway,The Original Prop Popper + HQ and Gemfan 31mm ...,NaN,
4,Track1,Yamasaur,CustomFPV,$30.00,Giveaway,RaceGOW3 Exclusive 65mm 2-banger Whoop Box,NaN,charlottesville.drone@gmail.com
5,Track1,FpvGaps,CustomFPV,$30.00,Giveaway,RaceGOW3 Exclusive 65mm 2-banger Whoop Box,NaN,
6,Track1,JANxD,Tiny Whoop,$25.00,Giveaway,CASH via PayPal,NaN,
7,Track1,GOONZ,FPVSkittles + Gemfan,$17.00,Giveaway,Mystery Draw + Gemfan Props and Extras,NaN,charlottesville.drone@gmail.com
8,Track1,Rodgsilva,Nick Burns,$20.00,Livestream,Must be present during livestream draw to win!...,NaN,


In [56]:
import tkinter as tk
from tkinter import ttk

# Create the main window
root = tk.Tk()
root.title("IGOW/RaceGOW Prize Automation App")
root.geometry("400x300")

# ===== Dropdown (ComboBox) =====
tk.Label(root, text="Select an option:").pack(pady=5)
options = week_dropdown_list
selected_option = tk.StringVar()
dropdown = ttk.Combobox(root, textvariable=selected_option, values=options, state="readonly")
dropdown.pack(pady=5)

# ===== Text Input =====
tk.Label(root, text="Enter text:").pack(pady=5)
text_entry = tk.Entry(root, width=30)
text_entry.pack(pady=5)

# ===== Output Label =====
output_label = tk.Label(root, text="", fg="blue")
output_label.pack(pady=10)

# ===== Button =====
def on_submit():
    selected_week = selected_option.get()
    winner_info = get_winner_info(week)
    winner_abbreviated = winner_info[['Winner','Prize Details','Email']]
    winner_abbreviated = winner_abbreviated.to_string(index=False) 
    text = text_entry.get()
    output_label.config(text=f"Generating emails for {selected_week} winners: {winner_abbreviated} .")

    # text_widget = tk.Text(root, height=10, width=50)
    # text_widget.pack()

    # Insert the table
    # text_widget.insert(tk.END, winner_abbreviated.to_string(index=False))


submit_button = tk.Button(root, text="Run", command=on_submit)
submit_button.pack(pady=10)

def on_close():
    root.quit()     # stop the mainloop
    root.destroy()  # destroy the window

root.protocol("WM_DELETE_WINDOW", on_close)
root.mainloop()


In [60]:
import tkinter as tk
from tkinter import ttk
import pandas as pd

# Create main window
root = tk.Tk()
root.title("IGOW/RaceGOW Prize Automation App")
root.geometry("1000x600")

# ===== Dropdown (ComboBox) =====
tk.Label(root, text="Select an option:").pack(pady=5)
selected_option = tk.StringVar()
dropdown = ttk.Combobox(root, textvariable=selected_option, values=week_dropdown_list, state="readonly")
dropdown.pack(pady=5)

# ===== Text Input =====
tk.Label(root, text="Enter text:").pack(pady=5)
text_entry = tk.Entry(root, width=30)
text_entry.pack(pady=5)

# ===== Output Label =====
output_label = tk.Label(root, text="", fg="blue")
output_label.pack(pady=5)

# ===== Treeview for Table =====
columns = ("Winner", "Prize Details", "Email")
tree = ttk.Treeview(root, columns=columns, show="headings", height=10)
for col in columns:
    tree.heading(col, text=col)
    tree.column(col, width=200)  # adjust width as needed
tree.pack(pady=5, fill=tk.X)

# Add vertical scrollbar
scrollbar = ttk.Scrollbar(root, orient="vertical", command=tree.yview)
tree.configure(yscrollcommand=scrollbar.set)
scrollbar.pack(side=tk.RIGHT, fill=tk.Y)

# ===== Button =====
def on_submit():
    # Clear previous rows
    for row in tree.get_children():
        tree.delete(row)
    
    selected_week = selected_option.get()
    winner_info = get_winner_info(selected_week)
    winner_abbreviated = winner_info[['Winner','Prize Details','Email']]
    
    # Insert rows into Treeview
    for _, row in winner_abbreviated.iterrows():
        tree.insert("", tk.END, values=list(row))
    
    output_label.config(text=f"Generating emails for {selected_week} winners:")

submit_button = tk.Button(root, text="Run", command=on_submit)
submit_button.pack(pady=10)

# ===== Close handler =====
def on_close():
    root.quit()
    root.destroy()

root.protocol("WM_DELETE_WINDOW", on_close)
root.mainloop()


Legion
regex
Daddio
Porkchop
bonsaihacker fpv
AzaleaFPV
nonne
Side FPV
rodgsilva
TDoge
